# C1 — Dự báo Doanh số Q2/2026
### Câu hỏi 1: Sản phẩm nào sẽ bán được? | DATA EXPLORERS 2026

**Mục tiêu:**
- Dự báo tổng doanh số tháng 4, 5, 6/2026
- Phân tích theo 5 nhóm sản phẩm
- Top 20 mẫu xe dự kiến bán chạy nhất

> Chạy **C0** trước để sinh `tnbike_data.csv`

# Thư viện và dữ liệu

In [ ]:
# thư viện và dữ liệu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import pmdarima as pm
from statsmodels.tsa.stattools import adfuller

# tải dữ liệu lịch sử (SQL + PDF T3/2026)
df        = pd.read_csv('tnbike_data.csv', parse_dates=['ds'])
future_df = pd.read_csv('future.csv',      parse_dates=['ds'])
df

In [ ]:
# gộp lịch sử + tương lai (Q2/2026)
df = pd.concat([df, future_df]).reset_index(drop=True)
df.tail()

In [ ]:
# đổi tên biến
df = df.rename(columns={'revenue': 'y'})
df.head(1)

In [ ]:
# biến ngày
df['ds'] = pd.to_datetime(df['ds'])
df.ds

# Ngày lễ

In [ ]:
# Tết Nguyên Đán
dates_tet = pd.to_datetime(df[df.is_tet == 1].ds)
tet = pd.DataFrame({"holiday":      "tet_nguyen_dan",
                    "ds":           dates_tet,
                    "lower_window": -7,
                    "upper_window": 7})

In [ ]:
# Ngày Quốc tế Thiếu nhi 1/6
dates_thieu_nhi = pd.to_datetime(df[df.is_thieu_nhi == 1].ds)
thieu_nhi = pd.DataFrame({"holiday":      "ngay_thieu_nhi",
                           "ds":           dates_thieu_nhi,
                           "lower_window": -14,
                           "upper_window": 3})

In [ ]:
thieu_nhi

In [ ]:
# gộp ngày lễ
holidays = pd.concat([tet, thieu_nhi])
holidays

In [ ]:
# bỏ cột ngày lễ khỏi df chính
df = df.drop(['is_tet', 'is_thieu_nhi'], axis=1)

In [ ]:
# tách tập train và tập dự báo (Q2/2026: T4, T5, T6)
training  = df.iloc[:-3, :]
future_df = df.iloc[-3:, :]

In [ ]:
# lấy tham số tốt nhất
parameters = pd.read_csv('Forecasting Product/best_params_prophet.csv', index_col=0)
parameters

In [ ]:
# trích xuất giá trị tham số
changepoint_prior_scale = float(parameters.loc['changepoint_prior_scale'][0])
holidays_prior_scale    = float(parameters.loc['holidays_prior_scale'][0])
seasonality_prior_scale = float(parameters.loc['seasonality_prior_scale'][0])
seasonality_mode        = parameters.loc['seasonality_mode'][0]

In [ ]:
# mô hình Prophet
m = Prophet(holidays=holidays,
            seasonality_mode=seasonality_mode,
            seasonality_prior_scale=seasonality_prior_scale,
            holidays_prior_scale=holidays_prior_scale,
            changepoint_prior_scale=changepoint_prior_scale)
m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
m.fit(training)

# Tính dừng

In [ ]:
# kiểm định ADF
pvalue = adfuller(x=training['y'].dropna())[1]
if pvalue < 0.05:
    print(f"Chuỗi thời gian DỪNG. P-Value = {pvalue:.3f}")
else:
    print(f"Chuỗi thời gian KHÔNG DỪNG. P-Value = {pvalue:.3f}")

In [ ]:
# lấy sai phân bậc 1
pvalue = adfuller(training['y'].diff().dropna())[1]
if pvalue < 0.05:
    print(f"DỪNG sau sai phân. P-Value = {pvalue:.3f}")
else:
    print(f"KHÔNG DỪNG sau sai phân. P-Value = {pvalue:.3f}")

# Mô hình SARIMAX (so sánh)

In [ ]:
# lấy tham số tốt nhất
params_s = pd.read_csv('Forecasting Product/best_params_sarimax.csv', index_col=0)
p = int(params_s.loc['p'][0]); d = int(params_s.loc['d'][0]); q = int(params_s.loc['q'][0])
P = int(params_s.loc['P'][0]); D = int(params_s.loc['D'][0]); Q = int(params_s.loc['Q'][0])

In [ ]:
# mô hình SARIMAX — tần suất tháng → seasonal = 12
model_s = pm.ARIMA(order=(p, d, q),
                   seasonal_order=(P, D, Q, 12),
                   suppress_warnings=True,
                   enforce_stationarity=False)
model_s.fit(training['y'])

# Dự báo

In [ ]:
# tạo dataframe tương lai
future = m.make_future_dataframe(periods=3, freq='MS')
future.tail()

In [ ]:
# dự báo Prophet
forecast = m.predict(future)
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(3)

In [ ]:
# vẽ biểu đồ thành phần
m.plot_components(forecast);

In [ ]:
# dự báo SARIMAX
predictions_sarimax = model_s.predict(n_periods=3)
print("SARIMAX Q2/2026:", predictions_sarimax)

In [ ]:
# trực quan hóa — thực tế vs dự báo
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(training['ds'], training['y']/1e9, label='Lịch sử', color='steelblue')
ax.plot(forecast.tail(3)['ds'], forecast.tail(3)['yhat']/1e9,
        label='Prophet', color='orange', marker='o')
ax.fill_between(forecast.tail(3)['ds'],
                forecast.tail(3)['yhat_lower']/1e9,
                forecast.tail(3)['yhat_upper']/1e9, alpha=0.2, color='orange')
ax.set_title('Dự báo Doanh số Q2/2026 — Thống Nhất Bike', fontsize=13)
ax.set_ylabel('Doanh số (tỷ VND)')
ax.set_xlabel('Tháng')
ax.legend()
plt.tight_layout()
plt.show();

# Đánh giá

In [ ]:
# kiểm định chéo Prophet
df_cv = cross_validation(m, horizon='90 days', period='30 days', initial='90 days')
performance_metrics(df_cv).head()

In [ ]:
# RMSE và MAPE
print(f"RMSE: {round(performance_metrics(df_cv)['rmse'].mean(), 0):,.0f}")
print(f"MAPE: {100*round(performance_metrics(df_cv)['mape'].mean(), 3):.1f}%")

In [ ]:
# sai số tăng theo thời gian?
plot_cross_validation_metric(df_cv, metric='rmse');

# Xuất kết quả

In [ ]:
# trích xuất dự báo Q2/2026
predictions_prophet = forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(3)
predictions_prophet

In [ ]:
# xuất kết quả
predictions_prophet.to_csv('Forecasting Product/Ensemble/predictions_prophet.csv', index=False)
print('✅ Đã lưu predictions_prophet.csv')